# 06 — Pure Pursuit Path Tracking

**Section:** Path Tracking · **Mirrors MATLAB:** *Path Following with Obstacle Avoidance*

Pure pursuit is a geometric controller that finds a point on the reference path at a fixed **look-ahead distance** ahead of the robot, then computes the curvature that would carry the robot to that point in a single arc.


## Intuition — what's actually going on?

You're driving a car. You see the road curving ahead of you. You don't steer aimed at the *closest* point on the road (you'd swerve constantly) — you aim at a point a comfortable distance **ahead** on the road, and steer toward that. That's pure pursuit.

The parameter that controls the feel is the **look-ahead distance** `Ld`. Short `Ld` means aggressive, twitchy tracking. Long `Ld` means smooth driving but you cut corners. Most production self-driving cars adapt `Ld` to speed (longer at highway speeds, shorter when parking).

The math turns out to be beautifully clean: aim a circular arc from your current pose to the look-ahead point, and the curvature of that arc is $\kappa = 2\sin(\alpha)/L_d$ where $\alpha$ is the angle to the look-ahead point. You command angular velocity $\omega = v \kappa$. That's it.


## Analytical derivation

**Geometric setup.** Place the robot at the origin with heading along $+\hat x$. The target point $(x_t, y_t)$ on the path is at distance $L_d$ from the robot:

$$x_t^2 + y_t^2 = L_d^2$$

The robot must trace a *circular arc* from its current pose to the target. Let $R$ be the radius of curvature of that arc. Geometrically, the perpendicular from the robot to the line $\text{(robot} \to \text{target)}$ has the chord property:

$$y_t = \frac{L_d^2}{2R}\quad\Longrightarrow\quad \kappa = \frac{1}{R} = \frac{2 y_t}{L_d^2}$$

Defining $\alpha$ as the angle from the robot heading to the target:

$$y_t = L_d \sin\alpha,\qquad \kappa = \frac{2 \sin\alpha}{L_d}$$

For a unicycle moving at velocity $v$ the relationship between curvature and angular velocity is $\omega = v\kappa$, giving the **pure-pursuit law**:

$$\boxed{\;\omega \;=\; \frac{2 v \sin\alpha}{L_d}\;}$$

**Choosing $L_d$.** Small $L_d$ → aggressive tracking but oscillation. Large $L_d$ → smooth but cuts corners. Common practice: $L_d \propto v$ (longer look-ahead at speed).

**Stability** (locally on a straight reference). Linearize the unicycle around $\theta = 0$, $y = 0$, constant $v$. With small $\alpha \approx -\theta + y/L_d$:
$$\dot y \approx v\theta,\qquad \dot\theta = \omega = \frac{2v\sin\alpha}{L_d} \approx \frac{2v}{L_d}\!\left(\frac{y}{L_d} - \theta\right).$$
Stacking $(y, \theta)$ gives a 2-state linear system with characteristic equation $s^2 + (2v/L_d)s + 2v^2/L_d^2 = 0$, roots in the open LHP for any $L_d > 0$ — *locally* stable in the basin $|\alpha| < \pi/2$. Outside that basin the controller can produce orbiting trajectories.

### Compatibility check — math ↔ code

| Math | Code |
|---|---|
| $\alpha = \arctan2(y_t - y_r,\ x_t - x_r) - \theta_r$ | `alpha = np.arctan2(tg[1]-x[1], tg[0]-x[0]) - x[2]` |
| Look-ahead point: scan path forward until $\|p_j - p_r\| \ge L_d$ | `while j < len(path)-1 and np.linalg.norm(path[j] - x[:2]) < Ld: j += 1` |
| $\omega = 2 v \sin\alpha / L_d$ | `omega = 2 * v * np.sin(alpha) / Ld` |
| Unicycle integration | `x = x + dt * np.array([v*cos, v*sin, omega])` |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sinusoidal reference path
xs_ref = np.linspace(0, 30, 400)
ys_ref = 3 * np.sin(xs_ref * 0.3)
path = np.column_stack([xs_ref, ys_ref])

x = np.array([0.0, -2.0, 0.0])      # x, y, theta
v = 1.5
Ld = 1.8
dt = 0.05
T = 25.0
N = int(T / dt)

hist = np.zeros((N, 3))
lookaheads = np.zeros((N, 2))


In [ ]:
for i in range(N):
    d = np.linalg.norm(path - x[:2], axis=1)
    j = int(np.argmin(d))
    while j < len(path) - 1 and np.linalg.norm(path[j] - x[:2]) < Ld:
        j += 1
    target = path[j]
    alpha = np.arctan2(target[1] - x[1], target[0] - x[0]) - x[2]
    omega = 2 * v * np.sin(alpha) / Ld

    x = x + dt * np.array([v * np.cos(x[2]), v * np.sin(x[2]), omega])
    hist[i] = x
    lookaheads[i] = target


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(xs_ref, ys_ref, 'b--', lw=1.5, label='Reference')
ax.plot(hist[:, 0], hist[:, 1], 'r-', lw=2, label='Robot')
ax.plot(0, -2, 'go', markersize=12, label='Start')
ax.legend(loc='upper right'); ax.grid(); ax.set_aspect('equal')
ax.set_title('Pure Pursuit Path Tracking')
plt.tight_layout()
plt.show()


## References & rigor notes

**Stability** (linearization on straight reference). Linearizing the unicycle-with-pure-pursuit closed loop around $\theta = 0$, $y = 0$, $v$ const, the cross-track error $y$ satisfies a 2nd-order ODE whose characteristic roots are in the open LHP for any $L_d > 0$. Damping ratio increases with $L_d$; small $L_d$ gives oscillatory tracking.

**Curvature limit.** Pure pursuit's effective tracking limit is $\min(1/L_d,\ \omega_\max/v)$: even when the commanded curvature would track a path tighter than $1/L_d$, the vehicle's max turn-rate $\omega_\max$ caps the achievable curvature. For sharp turns reduce $L_d$ (or use Stanley control, notebook 15).

**Cartan-distribution structure.** The constant-speed unicycle is a Cartan distribution on $SE(2)$ generated by the vector fields $X_1 = \cos\theta\,\partial_x + \sin\theta\,\partial_y$ (forward) and $X_2 = \partial_\theta$ (rotate). Chow's theorem (1939) gives small-time local controllability via $[X_1, X_2]$. Time-optimal paths under this distribution are Reeds-Shepp / Dubins curves.

**Production rule of thumb.** Many self-driving stacks use $L_d = \max(L_\min, k_v v)$ — adaptive look-ahead that scales with speed.

**References.**
- Coulter, R. C. (1992). *Implementation of the pure pursuit path tracking algorithm*. Technical Report CMU-RI-TR-92-01, Robotics Institute, Carnegie Mellon.
- Snider, J. M. (2009). *Automatic steering methods for autonomous automobile path tracking*. Technical Report CMU-RI-TR-09-08, Robotics Institute, Carnegie Mellon.
